# Reproducing Zaki et al. (2024) on DANDI:000718

**Paper.** Zaki, Pennington, Morales-Rodriguez et al., "Offline ensemble co-reactivation links memories across days", *Nature* 637, 145-155 (2025), [doi:10.1038/s41586-024-08168-4](https://doi.org/10.1038/s41586-024-08168-4).

**Dataset.** [DANDI:000718](https://dandiarchive.org/dandiset/000718/0.261122.2159), version **0.261122.2159**, which this attempt was asked to use (it is a repeat attempt, made blind, on the data as they are now). It is the most recent published version. The claims file's description was written against version 0.260825.1902; three NWB files differ between the two versions, and the data check below shows that the values this analysis uses are the same in both. Files are streamed from the archive; nothing is downloaded.

**Text read.** The claims file (`claims/zaki-2024.yaml`) was written from the full text of the version of record (PMC11666460): Results, figure and Extended Data legends, and Methods. The paper's analysis code (github.com/denisecailab/RetrospectiveMemoryLinkingAnalysis_2024) is not public (404 when this attempt was made). The source-data repository the paper names under Data availability is read, at commit a508164, only for the link claim and as a reference value for one claim.

**What the dataset holds.** Two of the nine mice of the calcium imaging and EEG/EMG experiment (Figure 5, Extended Data Figures 9 and 10): Ca_EEG2-1 (1.5 mA) and Ca_EEG3-4 (0.25 mA), with one offline recording for each offline period deposited (Ca_EEG2-1: the first offline recording after aversive encoding; Ca_EEG3-4: the first after neutral and the first after aversive encoding).

| Claim | Tested here |
|---|---|
| source-data-link | tested (link, with the paper's source data) |
| ed9b-aversive-freezing-high-vs-low | tested |
| offline-bursts-above-shuffle | tested |
| offline-skew-above-shuffle | tested |
| burst-locomotion-pause | tested |
| fig5c-wake-overlap-neutral-coburst | tested |
| fig5c-sleep-no-coburst-difference | blocked: no NREM or REM in the deposited offline recordings |
| fig5c-low-shock-wake-no-difference | blocked: no cell registration for the low-shock mouse |
| ed9c-eeg-cohort-linking | blocked: neutral and novel recall of the high-shock mouse not deposited |
| ed9-ed10-sleep-unchanged | blocked: sleep scoring outside the offline recordings, and the other mice, not deposited |
| fig1bc, fig1ef, fig1g, ed1n, ed2, ed3, ed4f, fig3c, fig3f, fig4b, fig4e, ed6, ed8, ed7, fig6a, fig6b | blocked: the mice of these experiments are not in the dataset |

In [ ]:
import zlib, time, resource
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import zaki

PAPER = "zaki-2024"
def seed(claim):
    return zlib.crc32(f"{PAPER}/{claim}".encode())
T_START = time.time()
print("dandiset", zaki.DANDISET, "version", zaki.VERSION)

## Data check

The assets of the pinned version, each file's subject, session and start time, and the arrays this analysis uses.

In [ ]:
A = zaki.assets()
rows = []
for path in sorted(A):
    if not path.endswith(".nwb"):
        continue
    f = zaki.open_h5(path)
    r = {"path": path.split("/")[-1], "subject": zaki.text(f["general/subject/subject_id"]),
         "session": zaki.text(f["general/session_id"]), "start": zaki.text(f["session_start_time"]),
         "size_GB": round(A[path]["size"] / 1e9, 2)}
    g = "processing/ophys/Fluorescence/Deconvolved/data"
    if g in f:
        ts = f["processing/ophys/Fluorescence/Deconvolved/timestamps"][()]
        r.update(frames=f[g].shape[0], cells=f[g].shape[1], ophys_Hz=round(1 / np.median(np.diff(ts)), 2),
                 ophys_s=round(ts[-1] - ts[0], 1))
    if "processing/behavior/MotionSeries" in f:
        ms = f["processing/behavior/MotionSeries"]
        r.update(motion_frames=ms["data"].shape[0], motion_Hz=float(ms["starting_time"].attrs["rate"]))
    if "acquisition/ActivitySignal" in f:
        a = f["acquisition/ActivitySignal"]
        r.update(telemetry_Hz=float(a["starting_time"].attrs["rate"]), telemetry_s=round(a["data"].shape[0] / float(a["starting_time"].attrs["rate"]), 1))
    if "processing/sleep/SleepIntervals" in f:
        s = zaki.table(f["processing/sleep/SleepIntervals"])
        r["sleep_states"] = ", ".join(f"{k} {np.sum(s['stop_time'][s['sleep_state']==k]-s['start_time'][s['sleep_state']==k]):.0f}s"
                                      for k in sorted(set(s["sleep_state"])))
    rows.append(r)
files = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(f"{len(A)} assets, {len(files)} NWB files, {len(A)-len(files)} external videos")
print(files.to_string())
print("\nsubjects:", files.subject.value_counts().to_dict())

In [ ]:
# Session descriptions name the context of each recall and the shock amplitude of each FC session.
for path in sorted(A):
    if path.endswith("image+ophys.nwb"):
        f = zaki.open_h5(path)
        print(path.split("/")[-1], "|", zaki.text(f["session_description"])[:200])

### Freezing, ensembles and value ranges

In [ ]:
FREEZE = {}
for path in sorted(A):
    if path.endswith("image+ophys.nwb"):
        f = zaki.open_h5(path)
        fr = zaki.freezing_readings(f)
        key = (zaki.text(f["general/subject/subject_id"]), zaki.text(f["general/session_id"]))
        FREEZE[key] = fr
        iv = zaki.table(f["processing/behavior/FreezingIntervals"])
        d = iv["stop_time"] - iv["start_time"]
        print(f"{key[0]:10s} {key[1]:16s} intervals {fr['n_intervals']:3d} (shortest {d.min():.2f} s), "
              f"motion {fr['motion'].min():.0f} to {fr['motion'].max():.0f} (n.a.), freezing (a) {fr['a_frames']:.2f}% (b) {fr['b_duration']:.2f}%")
print()
f = zaki.open_h5("sub-Ca-EEG2-1/sub-Ca-EEG2-1_ses-Recall1_image+ophys.nwb")
print(f["processing/behavior/FreezingIntervals"].attrs["description"].strip()[:330])

In [ ]:
# Cell registration of Ca_EEG2-1 (in its Week file), and the ensembles of its offline recording.
week = zaki.open_h5(zaki.WEEK)
regs = list(week["processing/cell_registration"].keys())
print(len(regs), "registration tables:", regs[0], "...", regs[-1])
reg = zaki.table(week["processing/cell_registration/OfflineDay2Session1vsConditioningSessions"])
print("columns:", list(reg))
OFF = {}
for name, path in zaki.OFFLINE.items():
    f = zaki.open_h5(path)
    X, ts, rois = zaki.deconvolved(f)
    Z, keep = zaki.zscore_cells(X)
    sl = zaki.table(f["processing/sleep/SleepIntervals"])
    act = f["acquisition/ActivitySignal"]
    OFF[name] = dict(X=X, ts=ts, rois=rois, Z=Z, keep=keep, sleep=sl, f=f,
                     act=act["data"][()], act_t0=float(act["starting_time"][()]), act_rate=float(act["starting_time"].attrs["rate"]))
    print(f"{name}: {X.shape[0]} frames x {X.shape[1]} cells, deconvolved {X.min():.1f} to {X.max():.1f} (n.a.), "
          f"{(~keep).sum()} cells without activity; frame step {np.diff(ts).min():.3f} to {np.diff(ts).max():.3f} s; "
          f"ROI ids {rois.min()}..{rois.max()} ({len(np.unique(rois))} unique); ActivitySignal {OFF[name]['act'].min():.3f} to {OFF[name]['act'].max():.3f}, "
          f"starts {OFF[name]['act_t0']:.3f} s; sleep rows {len(sl['start_time'])}, {sl['start_time'].min():.2f} to {sl['stop_time'].max():.2f} s")
o = OFF["Ca_EEG2-1 Day2 S1"]
ens = zaki.ensembles_from_registration(reg, "Ca_EEG2-1_OfflineDay2Session1", "Ca_EEG2-1_NeutralExposure", "Ca_EEG2-1_FC", o["rois"])
o["labels"] = ens["labels"]
print("registration rows with an offline cell:", ens["n_rows"], "unique offline ROI ids:", ens["n_unique"], "of", len(o["rois"]))
print("ensemble sizes:", pd.Series(ens["labels"]).value_counts().to_dict())
d = np.diff(np.r_[o["sleep"]["start_time"], o["sleep"]["stop_time"][-1]])
print("sleep interval durations (s), Ca_EEG2-1:", np.round(o["sleep"]["stop_time"] - o["sleep"]["start_time"], 2)[:10], "...")

In [ ]:
# What the variables are and which modules each offline file holds, from the files' own descriptions.
of = OFF["Ca_EEG2-1 Day2 S1"]["f"]
a = of["acquisition/ActivitySignal"]
print("ActivitySignal:", a.attrs["description"], "| unit:", a["data"].attrs["unit"], "| rate:", a["starting_time"].attrs["rate"], "Hz")
print("SleepIntervals:", of["processing/sleep/SleepIntervals"].attrs["description"])
print("sleep_state column:", " ".join(of["processing/sleep/SleepIntervals/sleep_state"].attrs["description"].split()))
for name in zaki.OFFLINE:
    f = OFF[name]["f"]
    print(f"{name}: processing modules {list(f['processing'])}; processing/ophys {list(f['processing/ophys'])}; "
          f"sleep states {sorted(set(OFF[name]['sleep']['sleep_state']))}")
    print("    session_description:", zaki.text(f["session_description"])[:400])
print("files with a cell_registration module:",
      [p.split("/")[-1] for p in sorted(A) if p.endswith(".nwb") and "cell_registration" in zaki.open_h5(p)["processing"]])

### Versions, duplicates and clocks

Three NWB files changed between version 0.260825.1902 (the claims file's description) and 0.261122.2159. The next cell compares, in each, the arrays this analysis reads.

In [ ]:
old = zaki.assets("0.260825.1902")
changed = [p for p in sorted(A) if p in old and old[p]["asset_id"] != A[p]["asset_id"]]
print("assets that differ:", changed)
check = ["processing/behavior/FreezingIntervals/start_time", "processing/behavior/FreezingIntervals/stop_time",
         "processing/behavior/MotionSeries/data", "processing/ophys/Fluorescence/Deconvolved/data",
         "processing/ophys/Fluorescence/Deconvolved/timestamps", "processing/ophys/Fluorescence/Denoised/rois",
         "processing/sleep/SleepIntervals/start_time", "processing/sleep/SleepIntervals/stop_time",
         "processing/sleep/SleepIntervals/sleep_state", "acquisition/ActivitySignal/data", "session_start_time", "session_description"]
for p in changed:
    fa, fb = zaki.open_h5(p, "0.260825.1902"), zaki.open_h5(p)
    for k in check:
        if k in fa and k in fb:
            x, y = fa[k][()], fb[k][()]
            if np.asarray(x).dtype.kind in "fiu":
                diff = float(np.max(np.abs(np.asarray(x, float) - np.asarray(y, float)))) if np.shape(x) == np.shape(y) else "shape differs"
            else:
                diff = "same" if np.array_equal(np.asarray(x), np.asarray(y)) else "differs"
            print(f"  {p.split('/')[-1]:45s} {k:55s} {str(np.shape(x)):14s} max abs difference / equality: {diff}")

In [ ]:
# Recordings deposited twice? Asset digests, and the arrays themselves for sessions of equal length.
dig = {p: zaki.asset_metadata(p)["digest"]["dandi:sha2-256"] for p in sorted(A)}
print("distinct SHA-256 digests:", len(set(dig.values())), "of", len(dig), "assets")
mot = {k: v["motion"] for k, v in FREEZE.items()}
keys = list(mot)
worst = 0
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        a, b = mot[keys[i]], mot[keys[j]]
        n = min(len(a), len(b))
        if abs(len(a) - len(b)) < 20:
            r = np.corrcoef(a[:n], b[:n])[0, 1]
            worst = max(worst, r)
            print(f"  motion {keys[i]} vs {keys[j]}: r = {r:.3f}")
print("largest motion correlation between sessions of equal length:", round(worst, 3))
n = 3000
for a_, b_ in [("Ca_EEG3-4 Day1 S1", "Ca_EEG3-4 Day2 S1"), ("Ca_EEG2-1 Day2 S1", "Ca_EEG3-4 Day1 S1"), ("Ca_EEG2-1 Day2 S1", "Ca_EEG3-4 Day2 S1")]:
    L = min(len(OFF[a_]["act"]), len(OFF[b_]["act"]))
    r = np.corrcoef(OFF[a_]["act"][:L], OFF[b_]["act"][:L])[0, 1]
    print(f"  ActivitySignal {a_} vs {b_}: r = {r:.3f}; cells {OFF[a_]['X'].shape[1]} vs {OFF[b_]['X'].shape[1]}")

**Clocks and sleep labels.** Five checks: (1) the behaviour clock, from the motion at the shock times the FC description gives (120, 180 and 240 s); (2) the imaging clock against the behaviour clock, from the cross-correlation of population activity and motion; (3) the imaging clock against the telemetry in the offline files, the same way with the ActivitySignal, and the sleep labels against the EMG with the labels shifted in time; (4) the session times of Ca_EEG2-1 against its continuous telemetry in the Week file; (5) the sleep labels of Ca_EEG2-1's offline recording against a scoring of the Week telemetry by the Methods' procedure.

In [ ]:
# (1) shocks in the FC sessions, against every other 2 s window of the session after the baseline
for key in [k for k in FREEZE if k[1] == "FC"]:
    fr = FREEZE[key]; t = fr["frame_times"]; m = fr["motion"]
    shock = [m[(t >= s) & (t < s + 2)].mean() for s in (120, 180, 240)]
    other = np.array([m[(t >= s) & (t < s + 2)].mean() for s in np.arange(122, 272, 2) if all(abs(s - x) >= 4 for x in (120, 180, 240))])
    shifted = [m[(t >= s + 10) & (t < s + 12)].mean() for s in (120, 180, 240)]
    print(f"{key[0]} FC motion in 2 s from each shock: {np.round(shock)}; same windows 10 s later: {np.round(shifted)}; "
          f"other 2 s windows median {np.median(other):.0f} (95th percentile {np.percentile(other, 95):.0f})")

# (2) imaging against behaviour: correlation of mean z-scored activity with motion, by lag
def lagged_corr(x, y, lags):
    out = []
    for L in lags:
        if L >= 0: out.append(np.corrcoef(x[L:], y[:len(y) - L])[0, 1])
        else: out.append(np.corrcoef(x[:L], y[-L:])[0, 1])
    return np.array(out)
lags_s = np.arange(-10, 10.5, 0.5)
for path in sorted(A):
    if not path.endswith("image+ophys.nwb"):
        continue
    f = zaki.open_h5(path)
    key = (zaki.text(f["general/subject/subject_id"]), zaki.text(f["general/session_id"]))
    X, ts, _ = zaki.deconvolved(f); Z, _ = zaki.zscore_cells(X)
    fr = FREEZE[key]
    # motion averaged in 0.5 s bins on the behaviour clock; activity in the same bins on the imaging clock
    edges = np.arange(0, min(ts[-1], fr["frame_times"][-1]), 0.5)
    mb = np.histogram(fr["frame_times"], edges, weights=fr["motion"])[0] / np.maximum(np.histogram(fr["frame_times"], edges)[0], 1)
    pb = np.histogram(ts, edges, weights=Z.mean(1))[0] / np.maximum(np.histogram(ts, edges)[0], 1)
    c = lagged_corr(pb, mb, (lags_s / 0.5).astype(int))
    print(f"{key[0]:10s} {key[1]:16s} r at lag 0 = {c[lags_s == 0][0]:+.3f}; largest |r| {c[np.argmax(np.abs(c))]:+.3f} at {lags_s[np.argmax(np.abs(c))]:+.1f} s; "
          f"median |r| at lags beyond 5 s {np.median(np.abs(c[np.abs(lags_s) > 5])):.3f}")

In [ ]:
# (3) offline files: imaging against telemetry, and sleep labels against EMG
for name, o in OFF.items():
    c_, mv = zaki.movement_bins(o["act"], o["act_t0"], o["act_rate"], 0.5)
    edges = np.r_[c_ - 0.25, c_[-1] + 0.25]
    pb = np.histogram(o["ts"], edges, weights=o["Z"].mean(1))[0] / np.maximum(np.histogram(o["ts"], edges)[0], 1)
    ok = np.histogram(o["ts"], edges)[0] > 0
    pb, mv2 = pb[ok], mv[ok]
    c = lagged_corr(pb, mv2, (lags_s / 0.5).astype(int))
    emg = o["f"]["acquisition/EMGSignal/data"][()]
    rate = o["act_rate"]
    # EMG RMS in 6 s epochs; wake against quiet wake with the labels as given and shifted
    ep = np.arange(0, len(emg) / rate - 6, 6.0)
    rms = np.array([np.sqrt(np.mean(emg[int(s * rate):int((s + 6) * rate)] ** 2)) for s in ep])
    res = {}
    for sh in (0, 30, 60, 120):
        st = zaki.state_at(((ep + 3 + sh) % (len(emg) / rate)), o["sleep"])
        w, q = rms[st == "wake"], rms[st == "quiet wake"]
        res[sh] = np.median(w) / np.median(q)
    print(f"{name}: activity x movement r at lag 0 = {c[lags_s == 0][0]:+.3f}, largest |r| {c[np.argmax(np.abs(c))]:+.3f} at {lags_s[np.argmax(np.abs(c))]:+.1f} s; "
          f"EMG RMS wake / quiet wake = " + ", ".join(f"{v:.2f} (labels shifted {k} s)" for k, v in res.items()))

In [ ]:
# (4) Ca_EEG2-1: session times against the Week file's continuous telemetry.
ep = zaki.table(week["intervals/epochs"])
epochs = pd.DataFrame({"session": ep["session_ids"], "start": ep["start_time"], "stop": ep["stop_time"]})
ref = zaki.text(week["session_start_time"])
epochs["clock"] = pd.Timestamp(ref) + pd.to_timedelta(epochs.start, unit="s")
print("Week file session_start_time:", ref)
print(epochs.sort_values("start").to_string())

# The offline file's telemetry is the Week telemetry from the epoch start: find the offset of best match.
fs = 500.0
W_eeg = week["acquisition/EEGSignal/data"]
st_off = float(epochs.start[epochs.session == "OfflineDay2Session1"].iloc[0])
seg = W_eeg[int((st_off - 5) * fs):int((st_off + 25) * fs)]
x = OFF["Ca_EEG2-1 Day2 S1"]["f"]["acquisition/EEGSignal/data"][:5000]
errs = [np.max(np.abs(seg[k:k + len(x)] - x)) for k in range(0, len(seg) - len(x))]
k = int(np.argmin(errs))
print(f"\nOffline file EEG matches the Week EEG at {k / fs - 5:+.3f} s from the epoch start (max abs difference {errs[k]:.2e} V)")

# Telemetry dropouts (EEG exactly constant over 10 s): the mouse off the home-cage receiver.
def dropouts(t0, t1):
    s = W_eeg[int(t0 * fs):int(t1 * fs)]
    nb = len(s) // 5000
    flat = np.ptp(s[:nb * 5000].reshape(nb, 5000), axis=1) == 0
    d = np.diff(np.r_[0, flat.astype(int), 0])
    return [(t0 + a * 10, t0 + b * 10) for a, b in zip(np.flatnonzero(d == 1), np.flatnonzero(d == -1))]
for sess in ["NeutralExposure", "FC", "Recall1"]:
    s0, s1 = epochs.loc[epochs.session == sess, ["start", "stop"]].iloc[0]
    for a, b in dropouts(s0 - 7200, s1 + 3600):
        print(f"{sess:16s} epoch {s0:.1f}-{s1:.1f} s: telemetry dropout {a:.0f}-{b:.0f} s, "
              f"i.e. {a - s0:+.0f} to {b - s0:+.0f} s from the session's start ({(b - a) / 60:.1f} min)")
    nxt = epochs[(epochs.start > s0 - 7200) & epochs.session.str.startswith("Offline")].sort_values("start")
    if len(nxt):
        print(f"{'':16s} first offline epoch in that window: {nxt.session.iloc[0]} at {nxt.start.iloc[0] - s0:+.0f} s from the session's start")
for name in ["FC"]:
    s0 = float(epochs.start[epochs.session == name].iloc[0])
    s = W_eeg[int(s0 * fs):int((s0 + 274) * fs)]; e = week["acquisition/EMGSignal/data"][int(s0 * fs):int((s0 + 274) * fs)]
    s2 = W_eeg[int((s0 - 3300) * fs):int((s0 - 3300 + 274) * fs)]
    print(f"\nTelemetry during the FC epoch itself: EEG SD {np.std(s):.2e} V, EMG SD {np.std(e):.2e} V (not a dropout); 55 min earlier, EEG SD {np.std(s2):.2e}")

In [ ]:
# (5) Sleep scoring of Ca_EEG2-1's Week telemetry by the Methods' procedure ("Sleep recordings and sleep scoring":
# 6 s epochs, a two-state Gaussian mixture on the EMG for sleep against wake, then one on the theta/delta ratio of
# the sleep epochs for REM against NREM), and the offline file's own labels against it.
t0 = time.time()
FEAT = zaki.epoch_features(week)
es = FEAT["epoch_s"]
STATE = zaki.score_sleep(FEAT)                    # theta/delta ratio as the Methods name it
STATE_LOG = zaki.score_sleep(FEAT, log_ratio=True)  # variant: its log
frac = lambda st: {s: round(100 * np.mean(st[st != "none"] == s), 1) for s in ("wake", "nrem", "rem")}
print(f"{len(STATE)} epochs of {es:.0f} s ({len(STATE) * es / 86400:.2f} days) read and scored in {time.time() - t0:.0f} s; "
      f"constant telemetry (mouse off the receiver) in {100 * FEAT['flat'].mean():.1f}% of epochs")
print("percent of scored epochs, ratio as given:", frac(STATE), "; log ratio:", frac(STATE_LOG))
print("theta/delta ratio of the sleep epochs, 5/25/50/75/95th percentiles:",
      np.percentile(FEAT["theta"][np.isin(STATE, ["nrem", "rem"])] / FEAT["delta"][np.isin(STATE, ["nrem", "rem"])], [5, 25, 50, 75, 95]).round(2))
# the deposited offline recording: every 6 s epoch wholly inside one of its SleepIntervals rows
st0 = float(epochs.start[epochs.session == "OfflineDay2Session1"].iloc[0])
o = OFF["Ca_EEG2-1 Day2 S1"]
pairs = [(lab_, STATE[i]) for s, e, lab_ in zip(o["sleep"]["start_time"], o["sleep"]["stop_time"], o["sleep"]["sleep_state"])
         for i in range(int(np.ceil((st0 + s) / es)), int((st0 + e) // es))]
print("\nCa_EEG2-1 OfflineDay2Session1, file's label (rows) against the Methods' scoring (columns), 6 s epochs:")
print(pd.crosstab(pd.Series([x for x, _ in pairs], name="file"), pd.Series([y for _, y in pairs], name="scored")).to_string())

**What the checks show.** The printed numbers above are read as follows (each statement is backed by a printed number).

- Counts: two subjects, 12 NWB files (Ca_EEG2-1: neutral exposure, FC, Recall1, one offline recording and the Week telemetry with 19 cell-registration tables; Ca_EEG3-4: five behaviour sessions and two offline recordings). Imaging is at about 15 Hz (the Methods: "temporally downsampled by 2 (to 15 Hz)"), behaviour video at 30 Hz. The telemetry is stored at 500 Hz, and its sample count matches the 600 s recording at that rate; the Methods say the HD-X02 implants "recorded ... at 100 Hz". The data are trusted on this.
- The offline recordings hold only wake and quiet wake; no NREM or REM. The sleep_state column defines four states: "quiet wake" ("awake and alert but at rest, without engaging in any voluntary movement"), "wake" ("capable of voluntary movement"), "sws" and "rem". So quiet wake is a state of waking; the paper's scoring (Methods, "Sleep recordings and sleep scoring") has wake, NREM and REM only.
- Scored by the Methods' procedure (6 s epochs, a two-state Gaussian mixture on log EMG RMS for sleep against wake), 64 of the 68 epochs that Ca_EEG2-1's offline file labels quiet wake are wake (2 NREM, 2 REM), as are all 6 labelled wake. The paper's own sleep/wake split therefore puts quiet wake in wake. Over the 9.36 days of the Week file this scoring gives 47.4% wake; the REM share of the rest depends on whether the theta/delta ratio or its log is fitted (10.9% or 21.1% of the time), so the REM split is not well determined.
- ActivitySignal is described as "the motion of the probe relative to the receiver", "a proxy for locomotion", unit "n.a."; it is signed and continuous (ranges above), and its absolute value is used as movement.
- The three offline files hold the processing modules ophys and sleep only; the one cell_registration module in the dataset is in Ca_EEG2-1's Week file. Ca_EEG3-4 has no cell registration.
- The session descriptions of the three offline files describe the one-hour webcam experiment ("a webcam on top of the bin ... Mouse behavior and calcium were recorded for an hour"), while the files hold 10-min recordings with home-cage telemetry and sleep labels, as the Methods describe for the EEG/EMG mice.
- The freezing intervals come from ezTrack with a minimum of 15 frames (0.5 s at 30 Hz); the shortest interval is printed above.
- The FC descriptions give the shock amplitude as "1.5" and "0.25" without a unit; the experiment description gives the two amplitudes in mA.
- The cell registration of Ca_EEG2-1 gives 97 neutral, 80 aversive, 166 overlap and 282 remaining cells in its offline recording, every one of its 625 ROIs once.
- The three files that changed since 0.260825.1902 hold the same values for every array used here (differences of 0 or float rounding).
- No two assets share a digest, and no two behaviour sessions of equal length have correlated motion; the two offline recordings of Ca_EEG3-4 hold different numbers of cells and uncorrelated telemetry. No recording is deposited twice.
- Behaviour clock: in Ca_EEG2-1's FC session, motion in the 2 s after each of the three shock times of the description (3387 to 3848) is far above every other 2 s window (95th percentile 843) and drops 10 s later, so its MotionSeries is on the session clock the description implies. In Ca_EEG3-4 (0.25 mA) the shocks raise motion too little to show this (366 to 841 against a median of 737); its behaviour clock is not established by this check.
- Imaging against behaviour: the correlation of population activity with motion peaks within 0.5 s of lag 0 in Ca_EEG2-1 Recall1 (+0.53) and in Ca_EEG3-4 FC, neutral exposure and Recall1 (-0.24 to -0.33), consistent with a shared clock. In Ca_EEG2-1 FC and neutral exposure and Ca_EEG3-4 Recall2 and Recall3 the peak is weak and away from 0 (+1.5 to +3 s, or -10 s); for these four sessions a shared clock is not established. No claim here uses imaging and behaviour together.
- Imaging against telemetry in the offline files: the correlation of population activity with |ActivitySignal| is positive at lag 0 (+0.03 to +0.15) but peaks 1.5 to 6.5 s later (activity after movement), and the EMG is higher in wake than quiet wake by a ratio that falls when the labels are shifted (1.16 against 0.96 to 1.11, 1.36 against 1.17 to 1.25, 1.05 against 1.00 to 1.03). These are consistent with one clock for imaging, telemetry and sleep labels but do not show it; the burst-locomotion test rests on it.
- Session times of Ca_EEG2-1: the offline file's EEG is the Week file's EEG from the epoch start (offset +0.002 s, difference 0 V), so the offline file's telemetry and the Week file share one clock. On that clock the telemetry drops out (EEG exactly constant: the mouse is off the home-cage receiver) from 3780 to 2560 s before the neutral exposure's start time, 3710 to 2930 s before FC's, and 3730 to 2850 s before Recall1's, and is normal during the FC epoch itself. Each session fits in its dropout if its true start is 1 h before its timestamp. By the timestamps, offline Day 2 sessions 1 and 2 start 47 and 17 min before the FC session they follow (2817 s before FC, 113 s after the dropout ends), and offline Day 1 sessions 1 to 4 before the neutral exposure. The behaviour sessions' start times are therefore about 1 h later than the offline recordings' and the telemetry's; the files do not say which is right. This does not change any result here, which use one session at a time.
- The Week file's start time is labelled +00:00, while its epochs fall at the -04:00 clock times of the session files (offline Day 2 session 1 at 09:24:27.677 in both).

## source-data-link

> The two deposited mice are the mice of the same names in the paper's per-mouse source data, with the same shock group, and the freezing computed from the dandiset's freezing intervals agrees with the per-mouse freezing the paper reports for them.

Criterion: matched by mouse name; the recall contexts from the session descriptions; freezing read as (a) the percentage of MotionSeries frames inside a freezing interval and (b) the summed interval duration over the MotionSeries duration. Reproduced when both mice are found, their shock amplitudes match the source data's groups, and all four values are within tolerance under one reading.

In [ ]:
ed9b = zaki.source_data_csv("ExtendedData9/ExtData9b.csv")
ed9c = zaki.source_data_csv("ExtendedData9/ExtData9c.csv")
subjects = sorted(set(files.subject))
print("mice in the dandiset:", subjects)
print("found in ExtData9b:", [s for s in subjects if s in set(ed9b.Mouse)], "; in ExtData9c:", [s for s in subjects if s in set(ed9c.Mouse)])
amp = {}
for s in subjects:
    f = zaki.open_h5([p for p in A if p.startswith(f"sub-{s.replace('_', '-')}") and "FC_image" in p and p.endswith(".nwb")][0])
    d = zaki.text(f["session_description"])
    amp[s] = d.split("foot shocks of ")[1].split(",")[0]
    print(f"{s}: FC description shock amplitude '{amp[s]}'; source data group {ed9b.Amplitude[ed9b.Mouse == s].iloc[0]} / {ed9c.Amplitude[ed9c.Mouse == s].iloc[0]}")
amp_match = all((amp[s] == "1.5") == (ed9b.Amplitude[ed9b.Mouse == s].iloc[0] == "High Shock") for s in subjects)
print("shock amplitudes match the groups:", amp_match)

# contexts of the recall sessions from their descriptions
ctx = {}
for path in A:
    if "Recall" in path and path.endswith(".nwb"):
        f = zaki.open_h5(path)
        d = zaki.text(f["session_description"])
        c = "Aversive" if "shock context" in d else "Novel" if "Novel context" in d else "Neutral" if "Neutral context" in d else "?"
        ctx[(zaki.text(f["general/subject/subject_id"]), zaki.text(f["general/session_id"]))] = c
print("recall contexts:", ctx)

expected = [("Ca_EEG2-1", "Aversive", 63.87, 6.39, ed9b), ("Ca_EEG3-4", "Aversive", 8.46, 0.85, ed9b),
            ("Ca_EEG3-4", "Neutral", 16.90, 1.69, ed9c), ("Ca_EEG3-4", "Novel", 14.60, 1.46, ed9c)]
rows = []
for mouse, c, val, tol, src in expected:
    sess = [k for k, v in ctx.items() if k[0] == mouse and v == c][0]
    fr = FREEZE[sess]
    srcval = src.Freezing[(src.Mouse == mouse) & (src.Context == c)].iloc[0]
    rows.append(dict(mouse=mouse, context=c, session=sess[1], source_data=round(srcval, 2), expected=val, tol=tol,
                     a_frames=round(fr["a_frames"], 2), b_duration=round(fr["b_duration"], 2),
                     a_ok=abs(fr["a_frames"] - val) <= tol, b_ok=abs(fr["b_duration"] - val) <= tol))
link = pd.DataFrame(rows)
print(link.to_string())
print("within tolerance: reading (a)", int(link.a_ok.sum()), "of 4; reading (b)", int(link.b_ok.sum()), "of 4")

**Result: reproduced.** Both mice are in the source data under the same names, their FC descriptions give 1.5 and 0.25 (the source data's high- and low-shock groups), and all four freezing values are within tolerance under both readings; under reading (b) the three values of Ca_EEG3-4 equal the source data to two decimals and Ca_EEG2-1 differs by 0.42 points.

## ed9b-aversive-freezing-high-vs-low

> In the calcium imaging and EEG/EMG experiment, high-shock mice froze more during aversive recall than low-shock mice (t7 = 8.99, p = 0.020).

Criterion: one mouse per group, so the paper's t-test cannot be run. Freezing as reading (a) in Recall1. Reproduced when Ca_EEG2-1 freezes more than Ca_EEG3-4 and the per-30-s-bin freezing of the two sessions (10 bins each) differs by a two-sided Mann-Whitney test at p < 0.05.

In [ ]:
hi, lo = FREEZE[("Ca_EEG2-1", "Recall1")], FREEZE[("Ca_EEG3-4", "Recall1")]
bh, bl = zaki.binned_freezing(hi), zaki.binned_freezing(lo)
mw = stats.mannwhitneyu(bh, bl, alternative="two-sided")
print(f"Recall1 freezing (a): Ca_EEG2-1 {hi['a_frames']:.2f}%, Ca_EEG3-4 {lo['a_frames']:.2f}%")
print("per-30-s bins Ca_EEG2-1:", np.round(bh, 1), "n =", len(bh))
print("per-30-s bins Ca_EEG3-4:", np.round(bl, 1), "n =", len(bl))
print(f"Mann-Whitney U = {mw.statistic:.1f}, two-sided p = {mw.pvalue:.2e}; bins where the high-shock mouse froze more: {np.sum(bh > bl)} of {len(bh)}")

**Result: reproduced.** The high-shock mouse froze in 64.1% of Recall1 frames and the low-shock mouse in 8.8%, and the high-shock mouse froze more in all 10 of the 30-s bins (Mann-Whitney p = 1.8e-04). Each group has one mouse here, against five and four in the paper, so this compares two animals, not two groups.

## offline-bursts-above-shuffle

> During offline periods, hippocampal population bursts occur more often than expected from shuffled neuronal activity, so CA1 neurons fire in a more coordinated way than chance.

Criterion: in each of the three offline recordings, the burst frequency (events per minute) of the population trace, against 1000 surrogates in which each cell's trace is circularly shifted by an independent random offset and the mean population trace recomputed (Extended Data Fig. 3l legend). The paper does not say how a surrogate's bursts are thresholded, so both readings its text allows are run: (a) the surrogate mean trace is z-scored again and thresholded at z = 2, as for the data, and (b) the surrogate mean trace is thresholded at the level that defines a burst in the true trace (the true mean trace's mean plus 2 SD). Reproduced when, under one reading, the true frequency exceeds the 95th percentile of the surrogates in all three; partially when, under one reading, it exceeds the surrogate mean in all three but the 95th percentile in fewer. The outcome is marked as resting on the reading that gives it.

The criterion named only reading (a) when this attempt was first made; a reader amended it to name both after a question from this attempt (`questions/zaki-2024.yaml`). Reading (a) uses the claim's generator, reading (b) a second generator (the claim's seed plus 1), each under its seed and the seed plus 1000 to 4000. The offsets are drawn uniformly over the whole recording (the legend does not give their range; a standard choice, see the record).


In [ ]:
def duration_min(ts):
    return len(ts) * np.median(np.diff(ts)) / 60

def run_shuffle(claim, measure_true, measure_sur, normalise=True, n=1000, offsets=(0, 1000, 2000, 3000, 4000)):
    out = {}
    for off in offsets:
        rng = np.random.default_rng(seed(claim) + off)
        for name, o in OFF.items():
            p = zaki.population_trace(o["Z"])
            sur = zaki.circular_shift_surrogates(o["Z"], rng, n, measure_sur(o, p), normalise=normalise)
            out[(off, name)] = dict(true=measure_true(o, p), mean=sur.mean(), p95=np.percentile(sur, 95),
                                    frac_ge=np.mean(sur >= measure_true(o, p)), sur=sur)
    return out

t0 = time.time()
freq_true = lambda o, p: zaki.burst_frequency(p, duration_min(o["ts"]))
B = run_shuffle("offline-bursts-above-shuffle", freq_true, lambda o, p: (lambda q: zaki.burst_frequency(q, duration_min(o["ts"]))))
# reading (b): surrogate mean trace thresholded at the true trace's mean + 2 SD (not z-scored again)
def raw_measure(o, p):
    m = o["Z"].mean(1); mu, sd = m.mean(), m.std()
    return lambda q: zaki.burst_frequency((q - mu) / sd, duration_min(o["ts"]))
Bb = run_shuffle("offline-bursts-above-shuffle", freq_true, raw_measure, normalise=False,
                 offsets=(1, 1001, 2001, 3001, 4001))  # second generator: seed + 1, and + 1000 to 4000
print(f"computed in {time.time() - t0:.0f} s")
for name in OFF:
    r = B[(0, name)]; v = Bb[(1, name)]
    print(f"{name}: true {r['true']:.2f} bursts/min; (a) re-z-scored surrogates mean {r['mean']:.2f}, 95th pct {r['p95']:.2f}, fraction >= true {r['frac_ge']:.3f}"
          f" | (b) true trace's threshold mean {v['mean']:.2f}, 95th pct {v['p95']:.2f}, max {v['sur'].max():.2f}, fraction >= true {v['frac_ge']:.3f}")
def outcome(res, off):
    above95 = sum(res[(off, n)]["true"] > res[(off, n)]["p95"] for n in OFF)
    abovem = sum(res[(off, n)]["true"] > res[(off, n)]["mean"] for n in OFF)
    return "reproduced" if above95 == 3 else "partially_reproduced" if abovem == 3 else "not_reproduced"
for lab, res, base in (("(a)", B, 0), ("(b)", Bb, 1)):
    for off in (0, 1000, 2000, 3000, 4000):
        print(f"{lab} seed + {base + off}: outcome {outcome(res, base + off)}; surrogate means " + ", ".join(f"{res[(base + off, n)]['mean']:.2f}" for n in OFF)
              + "; 95th pct " + ", ".join(f"{res[(base + off, n)]['p95']:.2f}" for n in OFF))


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 2.8))
for a, name in zip(ax, OFF):
    r = B[(0, name)]; v = Bb[(1, name)]
    a.hist(r["sur"], 30, color="0.6", label="(a) re-z-scored")
    a.hist(v["sur"], 30, color="C1", label="(b) true threshold"); a.axvline(r["true"], color="C0", ls="--")
    a.set_title(name, fontsize=9); a.set_xlabel("bursts per minute")
    print(f"{name}: surrogate range (a) {r['sur'].min():.2f} to {r['sur'].max():.2f}, (b) {v['sur'].min():.2f} to {v['sur'].max():.2f}; true {r['true']:.2f} (all inside the axes)")
ax[0].legend(fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
# Synthetic check of the two readings, on cells with no bursts (verifier, round 1). 600 independent AR(1) cells
# (coefficient 0.9) over 9000 frames (10 min at 15 Hz), alone and with a slow drift shared by all cells (a random
# walk smoothed over 60 s, at 4% of a cell's SD, which gives the mean trace a drift about as large as its noise).
# 200 surrogates of each reading; the claim's third generator (seed + 2).
rng = np.random.default_rng(seed("offline-bursts-above-shuffle") + 2)
T_, N_, n_sur, dur = 9000, 600, 200, 9000 / 15 / 60
noise = rng.standard_normal((T_, N_))
ar = np.empty((T_, N_)); ar[0] = noise[0]
for t in range(1, T_):
    ar[t] = 0.9 * ar[t - 1] + noise[t]
walk = zaki.zscore(np.convolve(np.cumsum(rng.standard_normal(T_ + 899)), np.ones(900) / 900, "valid"))
for lab_, D in (("independent cells", 0.0), ("with a shared slow drift", 0.04)):
    Zs, _ = zaki.zscore_cells(ar + D * walk[:, None] * ar.std())
    ps = zaki.population_trace(Zs); m_ = Zs.mean(1); mu, sd = m_.mean(), m_.std()
    true = zaki.burst_frequency(ps, dur)
    sa = zaki.circular_shift_surrogates(Zs, rng, n_sur, lambda q: zaki.burst_frequency(q, dur))
    sb = zaki.circular_shift_surrogates(Zs, rng, n_sur, lambda q: zaki.burst_frequency((q - mu) / sd, dur), normalise=False)
    sk = zaki.circular_shift_surrogates(Zs, rng, n_sur, zaki.skew)
    print(f"{lab_}: true {true:.2f} bursts/min; (a) surrogate mean {sa.mean():.2f}, 95th pct {np.percentile(sa, 95):.2f} -> "
          f"{'above' if true > np.percentile(sa, 95) else 'not above'}; (b) mean {sb.mean():.2f}, 95th pct {np.percentile(sb, 95):.2f} -> "
          f"{'above' if true > np.percentile(sb, 95) else 'not above'}; skew {zaki.skew(ps):.3f} against surrogate 95th pct {np.percentile(sk, 95):.3f}")

**Result: reproduced, under reading (b), an assumption.** With each surrogate thresholded at the true trace's own mean plus 2 SD, the true burst frequency (8.73, 6.42 and 5.31 bursts per minute) is above the 95th percentile of the surrogates (0.30, 0.10 and 0.00) and above every surrogate in all three recordings (fraction at or above the true value 0), under the second generator's seed and the four others. Under reading (a), with each surrogate z-scored again, the true frequency is below the surrogate mean in all three recordings (10.94, 8.81 and 8.48) under all five seeds; that reading alone would give not reproduced. The criterion accepts either reading, so the outcome rests on reading (b), which the paper does not confirm.

Reading (b) is a weak test. A circular shift keeps each cell's mean, so a surrogate mean trace has the true trace's mean and a smaller SD whenever the cells co-vary positively, and it rarely reaches the true trace's threshold. On synthetic cells with no bursts and only a slow drift shared by all of them, reading (b) puts the true frequency above the surrogates (5.90 against a 95th percentile of 2.60) and reading (a) below them (5.90 against a mean of 8.55), the pattern of the real data; on independent cells neither reading does (8.30 against 95th percentiles of 9.80 and 11.40). So the result under (b) shows that the cells co-vary positively, not that the co-variation comes as bursts. The reading-(a) result is the one that tests burst structure, and it does not hold here. A reader is asked whether reading (b) belongs in the criterion (`questions/zaki-2024.yaml`).

## offline-skew-above-shuffle

> The mean population activity during offline periods is more right-skewed than in shuffled data, showing brief burst-like activations that need coordinated activity of groups of neurons.

Criterion: as offline-bursts-above-shuffle, with the skewness of the population trace (scipy.stats.skew) in place of the burst frequency.

In [ ]:
t0 = time.time()
S = run_shuffle("offline-skew-above-shuffle", lambda o, p: stats.skew(p), lambda o, p: zaki.skew)
print(f"computed in {time.time() - t0:.0f} s; helper skew equals scipy.stats.skew:",
      np.isclose(zaki.skew(zaki.population_trace(OFF['Ca_EEG2-1 Day2 S1']['Z'])), stats.skew(zaki.population_trace(OFF['Ca_EEG2-1 Day2 S1']['Z']))))
for name in OFF:
    r = S[(0, name)]
    print(f"{name}: true skew {r['true']:.3f}; surrogates mean {r['mean']:.3f}, 95th pct {r['p95']:.3f}, max {r['sur'].max():.3f}, fraction >= true {r['frac_ge']:.3f}")
for off in (0, 1000, 2000, 3000, 4000):
    print(f"seed + {off}: outcome {outcome(S, off)}; surrogate means " + ", ".join(f"{S[(off, n)]['mean']:.3f}" for n in OFF)
          + "; 95th pct " + ", ".join(f"{S[(off, n)]['p95']:.3f}" for n in OFF))

**Result: reproduced.** In all three recordings the true skew of the population trace is above every one of the 1000 surrogates (fraction at or above the true value 0), well above their 95th percentile, under all five seeds. Skew does not depend on the scale of the trace, so the thresholding readings of the burst claim do not bear on it. Unlike reading (b) of the burst claim, this test is not passed by co-variation alone: on the synthetic cells with a shared slow drift the true skew is -0.276 against a surrogate 95th percentile of 0.105, and on independent cells 0.095 against 0.103.

## burst-locomotion-pause

> Offline population bursts coincide with the mouse briefly slowing down about 1 s before the burst and resuming its movement about 1 to 2 s later.

Criterion: movement is the absolute telemetry ActivitySignal averaged in 100 ms bins. Time zero is (a) the peak frame or (b) the first frame of the burst. For each burst with both windows inside the recording, mean movement from 1 s before to time zero against the mean from 5 to 3 s before, by a one-sided Wilcoxon signed-rank test (lower before the burst). Reproduced when p < 0.05 in all three recordings under one reading; partially when in one or two. The mean trace from -5 to +5 s is shown, with random times as a reference.

In [ ]:
rng = np.random.default_rng(seed("burst-locomotion-pause"))
lag = np.arange(-50, 50) * 0.1 + 0.05
LOC = {}
fig, ax = plt.subplots(1, 3, figsize=(12, 2.8))
for a, name in zip(ax, OFF):
    o = OFF[name]
    c, m = zaki.movement_bins(o["act"], o["act_t0"], o["act_rate"], 0.1)
    p = zaki.population_trace(o["Z"]); ev = zaki.burst_events(p)
    def windows(tz):
        if tz - 5 < c[0] or tz + 5 > c[-1]:
            return None
        return m[(c >= tz - 1) & (c < tz)].mean(), m[(c >= tz - 5) & (c < tz - 3)].mean(), np.interp(tz + lag, c, m)
    for rd, ix in (("a peak", 2), ("b onset", 0)):
        w = [windows(o["ts"][e[ix]]) for e in ev]; w = [x for x in w if x is not None]
        pre, base = np.array([x[0] for x in w]), np.array([x[1] for x in w])
        wt = stats.wilcoxon(pre, base, alternative="less")
        LOC[(name, rd)] = dict(n=len(w), dropped=len(ev) - len(w), pre=np.median(pre), base=np.median(base), p=wt.pvalue,
                               frac_lower=np.mean(pre < base), trace=np.mean([x[2] for x in w], axis=0))
    rt = rng.uniform(c[0] + 5, c[-1] - 5, 1000)
    wr = [windows(t) for t in rt]
    rpre, rbase = np.array([x[0] for x in wr]), np.array([x[1] for x in wr])
    LOC[(name, "random")] = dict(n=1000, pre=np.median(rpre), base=np.median(rbase), p=stats.wilcoxon(rpre, rbase, alternative="less").pvalue,
                                 frac_lower=np.mean(rpre < rbase), trace=np.mean([x[2] for x in wr], axis=0))
    for rd, col in (("a peak", "C0"), ("b onset", "C1"), ("random", "0.5")):
        a.plot(lag, LOC[(name, rd)]["trace"], color=col, label=rd)
    a.set_title(name, fontsize=9); a.set_xlabel("time from burst (s)"); a.set_ylabel("|ActivitySignal|")
ax[0].legend(fontsize=7); plt.tight_layout(); plt.show()
for (name, rd), r in LOC.items():
    tr = r["trace"]
    print(f"{name:18s} {rd:8s} n={r['n']:4d} median pre(-1..0 s) {r['pre']:.4f}, base(-5..-3 s) {r['base']:.4f}, pre<base in {100 * r['frac_lower']:.0f}%, one-sided p = {r['p']:.3f}; "
          f"mean trace at -5, -1, 0, +1, +2 s: {np.interp([-5, -1, 0, 1, 2], lag, tr).round(4)}; range {tr.min():.4f} to {tr.max():.4f}"
          + (f"; bursts dropped near the edges {r['dropped']}" if 'dropped' in r else ""))
for rd in ("a peak", "b onset"):
    print(rd, ": recordings with p < 0.05:", sum(LOC[(n, rd)]["p"] < 0.05 for n in OFF), "of 3")

In [ ]:
# Sensitivity to the clock and to the window lengths (verifier, round 1). The telemetry is moved by s seconds
# against the imaging (movement at telemetry time t placed at t + s), for s from -8 to +8 s and for each recording's
# lag of peak correlation of population activity with movement (data check, (3)). As a variant, the baseline window
# is shortened to 1 s (-4 to -3 s), the length of the window before the burst. Random times (the claim's second
# generator, seed + 1) give each test's reference.
def peak_lag(o):
    c_, mv = zaki.movement_bins(o["act"], o["act_t0"], o["act_rate"], 0.5)
    edges = np.r_[c_ - 0.25, c_[-1] + 0.25]
    cnt = np.histogram(o["ts"], edges)[0]
    pb = np.histogram(o["ts"], edges, weights=o["Z"].mean(1))[0] / np.maximum(cnt, 1)
    cc = lagged_corr(pb[cnt > 0], mv[cnt > 0], (lags_s / 0.5).astype(int))
    return float(lags_s[np.argmax(np.abs(cc))])
def pause_test(o, times, shift, base=(-5, -3)):
    c, m = zaki.movement_bins(o["act"], o["act_t0"], o["act_rate"], 0.1)
    c = c + shift
    w = [(m[(c >= t - 1) & (c < t)].mean(), m[(c >= t + base[0]) & (c < t + base[1])].mean())
         for t in times if t - 5 >= c[0] and t + 5 <= c[-1]]
    pre, b = np.array(w).T
    return len(w), stats.wilcoxon(pre, b, alternative="less").pvalue
rng = np.random.default_rng(seed("burst-locomotion-pause") + 1)
SH, EQ = [], []
for name, o in OFF.items():
    ev = zaki.burst_events(zaki.population_trace(o["Z"]))
    pk = peak_lag(o)
    rt = rng.uniform(o["ts"][0] + 5, o["ts"][-1] - 5, 1000)
    for s in [pk] + [x for x in np.arange(-8.0, 9.0, 1.0) if x != pk]:
        for rd, ix in (("a peak", 2), ("b onset", 0)):
            n, pv = pause_test(o, o["ts"][[e[ix] for e in ev]], s)
            SH.append(dict(recording=name, shift="peak lag %+.1f s" % pk if s == pk else "%+.0f s" % s, reading=rd, n=n, p=pv))
    for s in (0.0, pk):
        for base in ((-5, -3), (-4, -3)):
            n, pv = pause_test(o, o["ts"][[e[2] for e in ev]], s, base)
            nr, pr = pause_test(o, rt, s, base)
            EQ.append(dict(recording=name, shift=f"{s:+.1f} s", baseline=f"{base[0]} to {base[1]} s", bursts=n, p_bursts=round(pv, 3),
                           random_times=nr, p_random=round(pr, 3)))
SH = pd.DataFrame(SH)
for rd in ("a peak", "b onset"):
    t_ = SH[SH.reading == rd].pivot(index="shift", columns="recording", values="p")
    t_["recordings with p < 0.05"] = (t_ < 0.05).sum(axis=1)
    print(f"reading ({rd}): one-sided Wilcoxon p by shift of the telemetry\n", t_.round(3).to_string(), "\n")
print(pd.DataFrame(EQ).to_string(index=False))

**Result: not reproduced.** Under neither reading is movement in the second before a burst lower than 3 to 5 s before it in any recording (one-sided p 0.45 to 0.92). In all three recordings the mean |ActivitySignal| at -5 and -1 s around bursts is higher than around random times (0.13 to 0.36 against 0.08 to 0.21), while the medians are low throughout (0.03 to 0.10). Around random times movement in the later window is lower in Ca_EEG2-1 (p < 0.001), as movement declines over the recording.

The test assumes the imaging and telemetry share a clock, which the data check found consistent but could not show. With the telemetry moved to each recording's lag of peak correlation with population activity (+1.5, +5.5 and +6.5 s), p is 0.805, 0.982 and 0.863 (peak) and 0.798, 0.977 and 0.853 (onset), so the negative result survives that alignment. Over shifts from -8 to +8 s in 1 s steps, Ca_EEG3-4 Day 1 never reaches p < 0.05 (smallest 0.514), so no shift gives the result in all three recordings; at most two reach it, at -3 s (peak) and -3 and -4 s (onset), shifts that place movement earlier, against the direction of the correlation. With a 1 s baseline window (-4 to -3 s) in place of the criterion's 2 s, p is 0.809, 0.997 and 0.306 at lag 0, while random times give 0.009, 1.000 and 0.307, so the unequal windows of the criterion do not hide a pause. The only movement measure in the offline files is the telemetry ActivitySignal, described as the motion of the probe relative to the receiver; the paper's measure for Fig. 2d,e is webcam locomotion from the one-hour experiment, whose mice are not in the dataset.

## fig5c-wake-overlap-neutral-coburst

> After a strong (high-shock) aversive experience, during wake in the offline period the overlap ensemble co-bursts more with the neutral ensemble than with the aversive ensemble (t4 = 4.94, P = 0.016).

Criterion: the one high-shock mouse (Ca_EEG2-1) and its one deposited offline recording after aversive encoding (the first of 24). A burst takes the sleep state of its peak frame. Wake read as (a) wake and quiet wake, (b) wake alone; co-participation read as (i) both ensembles participate, (ii) both and not the third. Over the wake bursts, the percentage with overlap-neutral (OxN) and overlap-aversive (OxA) co-participation. Reproduced when OxN > OxA under one reading and an exact McNemar test gives p < 0.05; partially when OxN > OxA under one reading without that; not reproduced when OxN <= OxA under every reading.

In [ ]:
o = OFF["Ca_EEG2-1 Day2 S1"]
lab = o["labels"][o["keep"]]
tr = zaki.ensemble_traces(o["Z"], lab)
p = zaki.population_trace(o["Z"]); ev = zaki.burst_events(p)
part = zaki.coparticipation(ev, tr)
state = zaki.state_at(o["ts"][[e[2] for e in ev]], o["sleep"])
print("bursts:", len(ev), "by state of the peak frame:", pd.Series(state).value_counts().to_dict())
print("ensemble participation over all bursts (%):", {k: round(100 * v.mean(), 1) for k, v in part.items()})
O, N, Av = part["overlap"], part["neutral"], part["aversive"]
CO = []
for wname, wmask in (("a wake+quiet wake", np.isin(state, ["wake", "quiet wake"])), ("b wake alone", state == "wake")):
    for cname, (oxn, oxa) in (("i both", (O & N, O & Av)), ("ii both, not third", (O & N & ~Av, O & Av & ~N))):
        b = int(np.sum(oxn & ~oxa & wmask)); cc = int(np.sum(oxa & ~oxn & wmask))
        pv = stats.binomtest(b, b + cc, 0.5).pvalue if b + cc else 1.0
        CO.append(dict(wake=wname, coparticipation=cname, bursts=int(wmask.sum()), OxN=round(100 * oxn[wmask].mean(), 1),
                       OxA=round(100 * oxa[wmask].mean(), 1), only_OxN=b, only_OxA=cc, mcnemar_p=round(pv, 3)))
CO = pd.DataFrame(CO); print(CO.to_string())
# Reference: the paper's per-mouse value for this mouse over its 24 recordings (source data, Fig_5c.csv)
f5 = zaki.source_data_csv("Figure5/Fig_5c.csv")
print("\nsource data, Ca_EEG2-1:\n", f5[f5.Mouse == "Ca_EEG2-1"][["SleepState", "EnsembleCombo", "Fraction"]].to_string(index=False))
# Non-burst reference: ensemble co-activation over all frames outside bursts, wake (a)
inb = np.zeros(len(p), bool)
for s, e, _ in ev: inb[s:e + 1] = True
fstate = zaki.state_at(o["ts"], o["sleep"])
wf = np.isin(fstate, ["wake", "quiet wake"]) & ~inb
print(f"\noutside bursts, frames in wake (a): {wf.sum()}; OxN frames {100 * np.mean((tr['overlap'] > 2) & (tr['neutral'] > 2) & wf) / wf.mean():.2f}%, "
      f"OxA frames {100 * np.mean((tr['overlap'] > 2) & (tr['aversive'] > 2) & wf) / wf.mean():.2f}%")

**Result: partially reproduced.** Under wake alone (reading b), OxN exceeds OxA in both co-participation readings (33.3% against 29.2% with both participating; 8.3% against 4.2% with the third ensemble absent), but over 24 wake bursts the McNemar test is far from significant (2 against 1 discordant bursts, p = 1.0). Under wake and quiet wake together (reading a, 85 bursts) the order reverses (25.9% against 35.3%, and 8.2% against 17.6%; 7 against 15 discordant bursts, p = 0.134). The criterion's partial outcome is met under reading (b) alone, on 3 discordant bursts.

Reading (a) is the better match to the paper: the file defines quiet wake as a waking state ("awake and alert but at rest"), the Methods' sleep/wake split scores 64 of its 68 quiet-wake epochs as wake (data check), and the paper places this co-reactivation "during brief periods of quiet wake" (Results, "Co-reactivation occurs more during wake"). Under that reading OxN is below OxA. The outcome is kept as the criterion gives it, and a reader is asked whether reading (b) belongs in the criterion (`questions/zaki-2024.yaml`). The paper's value for this mouse, over all 24 recordings, is 10.4% against 8.4% in wake (source data). One of the mouse's 24 offline recordings and one of the paper's five high-shock mice are tested here, z-scored over this one recording.

## Blocked claims

- **fig5c-sleep-no-coburst-difference**: the deposited offline recordings hold wake and quiet wake only (data check); the NREM and REM epochs of the paper's 24 recordings per mouse are in the 23 recordings not deposited.
- **fig5c-low-shock-wake-no-difference**: the low-shock mouse Ca_EEG3-4 has no cell registration in the dataset, so its offline cells cannot be assigned to ensembles; and one of four low-shock mice is too few for the paper's paired test.
- **ed9c-eeg-cohort-linking**: the high-shock mouse Ca_EEG2-1 has no neutral or novel recall in the dataset; the low-shock mouse alone cannot test a group difference; its values are printed below.
- **ed9-ed10-sleep-unchanged**: needs the sleep scoring of whole days and the no-Miniscope mice; the dataset holds sleep labels only inside the offline recordings, and one week of unscored telemetry for one mouse, scored below by the Methods' procedure as a description of one mouse.
- **The one-hour offline experiments, behaviour-only, chemogenetic, cocaine, inhibitory tagging and no-shock experiments** (16 claims): none of their mice is in the dataset.

In [ ]:
# ed9c-eeg-cohort-linking, descriptive, the low-shock mouse only (the claim stays blocked: the high-shock mouse has no
# neutral or novel recall here, and the paper's test is a paired t-test over 4 low-shock mice).
for c_ in ("Neutral", "Novel"):
    k_ = [k for k, v in ctx.items() if k[0] == "Ca_EEG3-4" and v == c_][0]
    print(f"Ca_EEG3-4 {c_:8s} recall ({k_[1]}): freezing (a) {FREEZE[k_]['a_frames']:.2f}%, (b) {FREEZE[k_]['b_duration']:.2f}%")

In [ ]:
# ed9-ed10-sleep-unchanged, descriptive, one mouse (the claim stays blocked). Sleep features of Ca_EEG2-1 in the 24 h
# before neutral encoding and the 24 h after neutral and after aversive encoding (Extended Data Fig. 10a: "the 24 h
# prior to the start of the first Neutral encoding experience, and ... the 24 hrs following Neutral encoding and after
# Aversive encoding"). The mouse is away from its home cage, and its telemetry constant, while it is in the
# behaviour room; the encoding period is taken as that dropout (data check, (4)), not the session's timestamp, which
# is about 1 h later on this clock.
flat = FEAT["flat"]
d_ = np.diff(np.r_[0, flat.astype(int), 0])
runs = [(a * es, b * es) for a, b in zip(np.flatnonzero(d_ == 1), np.flatnonzero(d_ == -1)) if b - a >= 10]
def away(sess):
    s0 = float(epochs.start[epochs.session == sess].iloc[0])
    return [r for r in runs if s0 - 7200 <= r[0] <= s0 + 600][0]
nA, nB = away("NeutralExposure"); fA, fB = away("FC")
print(f"away for neutral encoding {nA:.0f}-{nB:.0f} s, for aversive encoding {fA:.0f}-{fB:.0f} s")
W = {"pre-neutral": (nA - 86400, nA), "post-neutral": (nB, nB + 86400), "post-aversive": (fB, fB + 86400)}
sl_ = lambda st, a, b: st[int(a // es):int(b // es)]
for lab_, st in (("theta/delta as given", STATE), ("log theta/delta", STATE_LOG)):
    tab = pd.DataFrame({k: zaki.sleep_features(sl_(st, a, b), es) for k, (a, b) in W.items()})
    print(f"\n{lab_}:\n", tab.round(2).to_string())
print("\nscored hours per window:", {k: round(np.sum(sl_(STATE, a, b) != "none") * es / 3600, 2) for k, (a, b) in W.items()})
print("percent wake in the first four 30-min bins after encoding, and at the same clock time on the day before neutral encoding:")
for lab_, t_ in (("post-neutral", nB), ("post-aversive", fB), ("same time, 24 h before post-neutral", nB - 86400)):
    print(f"  {lab_:38s}", [round(100 * np.mean(x[x != "none"] == "wake"), 1) for x in (sl_(STATE, t_ + i * 1800, t_ + (i + 1) * 1800) for i in range(4))])
# Extended Data Fig. 9g: the recording scheme against the whole offline period (day 2, the 19 imaging epochs)
off_ = epochs[epochs.session.str.startswith("OfflineDay2")]
inside = np.zeros(len(STATE), bool)
for s, e in zip(off_.start, off_.stop):
    inside[int(s // es):int(e // es)] = True
span = STATE[int(off_.start.min() // es):int(off_.stop.max() // es)]
print(f"\nday 2: {len(off_)} imaging epochs, state (%):", frac(STATE[inside]), "; whole span from the first to the last:", frac(span))

**Result: both claims stay blocked; descriptive values for one mouse.** The low-shock mouse froze 17.40% (a) or 16.90% (b) in neutral recall and 15.06% or 14.60% in novel recall, a difference of about 2 points; the paper's test is over four low-shock and five high-shock mice. For Ca_EEG2-1, scored by the Methods' procedure, the time asleep is 47.81% in the 24 h before neutral encoding, 53.12% after it and 57.15% after aversive encoding, with mean wake bouts of 88, 83 and 60 s; NREM bouts are 33 to 35 s under the ratio as given. The REM figures (REM bouts of about 10 s, and 40 to 46% of wake bouts followed by REM) are not physiological, so this scoring does not determine REM. In the first 30 min back in the home cage the mouse is awake 67% (after neutral) and 49% (after aversive) of the time, against 93% at the same clock time the day before. On day 2 the 19 imaging epochs hold 27.4% wake, 59.7% NREM and 12.9% REM, against 25.8%, 59.9% and 14.3% over the whole span they sample, in line with Extended Data Fig. 9g. With one mouse none of the paper's tests (across 9 mice, and against 4 mice without a Miniscope, which are not in the dataset) can be run.

## Summary

In [ ]:
summary = pd.DataFrame([
    ("source-data-link", "reproduced", "4 of 4 freezing values within tolerance under both readings"),
    ("ed9b-aversive-freezing-high-vs-low", "reproduced", f"64.1% against 8.8%, Mann-Whitney p = {mw.pvalue:.1e} (one mouse per group)"),
    ("offline-bursts-above-shuffle", "reproduced (assumed)", "above the 95th pct in 3 of 3 under reading (b); below the mean in 3 of 3 under (a)"),
    ("offline-skew-above-shuffle", "reproduced", "true skew above every surrogate in 3 of 3"),
    ("burst-locomotion-pause", "not_reproduced", "no pre-burst slowing in 0 of 3 recordings under either reading, also at the lag of peak correlation"),
    ("fig5c-wake-overlap-neutral-coburst", "partially_reproduced", "OxN > OxA under wake alone, McNemar p = 1.0; OxN < OxA with quiet wake"),
    ("fig5c-sleep-no-coburst-difference", "blocked", "data_missing"),
    ("fig5c-low-shock-wake-no-difference", "blocked", "data_missing"),
    ("ed9c-eeg-cohort-linking", "blocked", "data_missing (low-shock mouse shown descriptively)"),
    ("ed9-ed10-sleep-unchanged", "blocked", "data_missing (one mouse scored descriptively)"),
    ("16 claims on other experiments", "blocked", "data_missing"),
], columns=["claim", "outcome", "basis"])
print(summary.to_string(index=False))
print(f"\nnotebook run time {(time.time() - T_START) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** Two of nine mice of the EEG/EMG cohort and one offline recording per offline period (one of 24 for the high-shock mouse) are tested, so no group statistic of the paper can be run; the criteria compare recordings or bursts instead. The shuffle and burst claims, which the paper shows on the eight mice of the one-hour experiment (Extended Data Fig. 3), are tested on the EEG/EMG mice's offline recordings. Movement is the telemetry ActivitySignal, not the webcam locomotion of Fig. 2d,e. Circular-shift offsets are drawn uniformly over the recording (a standard choice). Wake is read with and without the dataset's quiet wake, a state the paper does not name. The paper does not say how a shuffled trace is thresholded; the burst claim is reproduced only when it is held to the true trace's threshold (reading b) and not when it is z-scored again (reading a), so its outcome rests on that reading, and a synthetic check shows that reading (b) is passed by cells that only share a slow drift.

**What was found about the dataset.**

- Ca_EEG2-1's behaviour sessions are timestamped about 1 h after the telemetry shows the mouse away from its home-cage receiver, and its first offline recording after aversive encoding starts 47 min before the FC session's start time; the offline recording and its telemetry share one clock (the offline file's EEG is the Week file's at the epoch start), and the behaviour sessions sit on another, about 1 h later.
- The Week file's session start time is labelled UTC (+00:00), but its epochs fall at the local (-04:00) clock times of the session files.
- The telemetry is stored at 500 Hz; the Methods say 100 Hz.
- The offline files' session descriptions describe the one-hour webcam experiment, not the 10-min home-cage recordings with telemetry that they hold.
- The sleep labels include a quiet wake state, defined in the column's description as a waking state, which the paper's scoring does not have; the Methods' sleep/wake split scores it as wake.
- The FC descriptions give the shock amplitude without a unit.
- The dataset holds 2 of the 9 mice, 1 of the 24 offline recordings of Ca_EEG2-1 (19 registration tables, for recordings not deposited), no cell registration for Ca_EEG3-4, no NREM or REM epochs, and none of the mice of the paper's other experiments.
- Ensemble sizes, freezing and the subjects' shock groups agree with the paper's source data; no recording is deposited twice; the three files changed since 0.260825.1902 hold the same values.